# 04 · GenAI Tools Demo

**Workshop:** AI for Actuaries — From Foundations to AI Agents
**Session / Part:** S2.P2
**Slides covered:** S2.P2.4, S2.P2.6, S2.P2.8
**Author:** Dr Rohan Yashraj Gupta (FIA, FIAI), with Satya Sai Mudigonda and Sai Krishna Vadali
**Workshop date:** 15 May 2026 · Hilton near Airport, Mumbai
**License:** CC BY-NC 4.0 — for educational use within the IFoA workshop and follow-up case study

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rohanyashraj/ifoa-workshop/blob/main/notebooks/04_genai_tools.ipynb)

## What this notebook does

Three reproducible Gemini API prompts that mirror Demos 1, 2, and 3 from the slide deck.
Each demo saves its output to a file under `/content/` so you have artefacts to refer
back to during Part 2 of Session 2 and the case study.

| Demo | LOB    | Output                                              | File                          |
|------|--------|-----------------------------------------------------|-------------------------------|
| 1    | GI     | 10 motor rating factors with direction & rationale  | `/content/demo_1_output.md`   |
| 2    | Life   | UW guidelines for ABC Term Life (markdown)          | `/content/demo_2_output.md`   |
| 3    | Health | 200-word board summary of a hospitalisation model   | `/content/demo_3_output.md`   |

Demo 4 (Cursor IDE) is a live screen-share, not a notebook cell.

## Prerequisites

- Google account (for Colab)
- Gemini API key — set in **Colab Secrets** as `GOOGLE_API_KEY` (free tier is sufficient)
- No local install required

## How to run

Top menu → **Runtime → Run all**. The first cell installs `google-genai`; the rest run without intervention.


## 0. Install

In [1]:
# Install google-genai. Pinned at 1.0+ — confirm against your Colab runtime
# at notebook freeze time.
!pip install -q google-genai

## 0.1 Standard imports and reproducibility

In [2]:
# === Standard imports ===
import json
import os
import datetime as dt
from pathlib import Path

# Reproducibility (LLM outputs are still non-deterministic by default —
# we set temperature=0 in calls below to get closer-to-stable results).
SEED = 42

# Notebook-specific imports below
# -------------------------------------------------------------
from google import genai
from google.genai import types

## 0.2 API setup

We read the Gemini API key from **Colab Secrets**. Add it in the left sidebar (key icon)
under the name `GOOGLE_API_KEY`. Local fallback: set the environment variable.

Pinning the model identifier matters — `gemini-2.5-flash` today is not the same model
in six months. Always pin in code.


In [3]:
# Read GOOGLE_API_KEY from Colab Secrets, fall back to env var.
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    if "GOOGLE_API_KEY" not in os.environ:
        raise RuntimeError(
            "GOOGLE_API_KEY is not set. "
            "Add it via Colab Secrets (left sidebar key icon) "
            "or set the env variable."
        )

# Pin the model. Confirm at notebook-finalisation time and update if the
# stable identifier has changed.
MODEL_ID = "gemini-2.5-flash"

# Output directory.
OUT_DIR = Path("/content")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Single shared client.
client = genai.Client()

print(f"API key loaded. Model pinned to: {MODEL_ID}")


API key loaded. Model pinned to: gemini-2.5-flash


## 0.3 Prompt log

Per slide S2.P2.12: log every API call. The helper below appends a one-line
record to `/content/prompt_log.csv` for audit and reproducibility.


In [4]:
LOG_PATH = OUT_DIR / "prompt_log.csv"

def _log_call(demo_id: str, prompt: str, response_text: str) -> None:
    """Append a one-line record of the API call to the prompt log."""
    ts = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")
    header_needed = not LOG_PATH.exists()
    with LOG_PATH.open("a", encoding="utf-8") as f:
        if header_needed:
            f.write("timestamp_utc,demo_id,model,prompt_chars,response_chars\n")
        f.write(
            f"{ts},{demo_id},{MODEL_ID},"
            f"{len(prompt)},{len(response_text)}\n"
        )

print(f"Logger ready. Records will be appended to: {LOG_PATH}")


Logger ready. Records will be appended to: /content/prompt_log.csv


## 1. Demo 1 — risk-factor table for ABC Motor (GI)

**Slide:** S2.P2.4
**Goal:** Generate ten rating factors for an ABC Motor private car comprehensive
product, each tagged with directional impact and a one-line justification.
**Output format:** JSON, saved as a fenced block inside `/content/demo_1_output.md`.
The raw JSON is also persisted to `/content/demo_1_output.json` for downstream use.

> ABC Insurer is hypothetical — see `00_personas_and_datasets.md`. Numbers are
> illustrative.


In [5]:
# --- Demo 1 ---------------------------------------------------------------
DEMO_1_PROMPT = """You are advising the pricing team at ABC General, a hypothetical
mid-sized Indian motor insurer. List 10 rating factors that should drive the
pure premium for private car comprehensive cover in India.

For each factor, return:
  - factor: short name
  - direction: "increase" or "decrease" (effect on premium when the factor goes up)
  - justification: a single sentence, no more than 25 words

Constraints:
  - Use factors that an Indian motor insurer can realistically observe today.
  - Do not include telematics or annual mileage unless they are commonly captured.
  - Return strictly the JSON array, nothing else.

Output schema:
[{"factor": str, "direction": str, "justification": str}, ...]
"""

# Structured-output mode guarantees a parseable JSON response.
resp = client.models.generate_content(
    model=MODEL_ID,
    contents=DEMO_1_PROMPT,
    config=types.GenerateContentConfig(
        temperature=0.0,
        response_mime_type="application/json",
    ),
)

factors = json.loads(resp.text)

# Quick sanity check.
assert isinstance(factors, list), "Expected a JSON array"
assert len(factors) >= 8, f"Expected ~10 factors, got {len(factors)}"
for f in factors:
    assert {"factor", "direction", "justification"} <= f.keys()

# Persist: pretty markdown for human reading + raw json for downstream code.
demo_1_md_path   = OUT_DIR / "demo_1_output.md"
demo_1_json_path = OUT_DIR / "demo_1_output.json"

md = [
    "# Demo 1 · ABC Motor — rating factor draft",
    "",
    f"_Generated by `{MODEL_ID}` at "
    f"{dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds')} UTC._",
    "",
    "_Illustrative — ABC Insurer is hypothetical. Verify factors and magnitudes "
    "against a live GLM run before any pricing decision._",
    "",
    "## Factors",
    "",
    "```json",
    json.dumps(factors, indent=2, ensure_ascii=False),
    "```",
    "",
]
demo_1_md_path.write_text("\n".join(md), encoding="utf-8")
demo_1_json_path.write_text(json.dumps(factors, indent=2, ensure_ascii=False), encoding="utf-8")

# Show what came back.
print(f"Got {len(factors)} factors. First 3:")
for f in factors[:3]:
    print(f"  - {f['factor']:<25s}  {f['direction']:<8s}  {f['justification']}")

_log_call("demo_1", DEMO_1_PROMPT, resp.text)
print()
print(f"Output saved to {demo_1_md_path}")


Got 10 factors. First 3:
  - Insured Declared Value (IDV)  increase  Higher IDV implies greater potential loss in case of total loss or significant damage, leading to higher claim payouts.
  - Vehicle Age                decrease  Older vehicles generally have lower Insured Declared Value (IDV), reducing the insurer's maximum liability for total loss.
  - Geographical Zone          increase  Urban and high-density zones typically experience higher accident frequency, theft rates, and repair costs, increasing claim likelihood and severity.

Output saved to /content/demo_1_output.md


## 2. Demo 2 — UW guidelines for ABC Term Life (Life)

**Slide:** S2.P2.6
**Goal:** Draft underwriting guidelines for an ABC Term Life product covering
issue ages 25–55 and sums assured up to ₹1 crore (Indian retail market).
**Output format:** Markdown, 300–400 words, saved to `/content/demo_2_output.md`.

> Output is a draft. Numerical thresholds (NoMed/TeleMed/FullMed bands etc.) must
> be matched against ABC Life's live underwriting matrix before publication.


In [6]:
# --- Demo 2 ---------------------------------------------------------------
DEMO_2_PROMPT = """You are drafting underwriting guidelines for ABC Life, a
hypothetical Indian life insurer. Product: ABC Term Life, level term cover.

Parameters:
  - Issue ages: 25-55 (last birthday)
  - Sums assured: up to INR 1 crore
  - Market: Indian retail
  - Distribution: agency and direct online

Required sections:
  1. Eligibility & age bands
  2. Medical underwriting tiers (NoMed / TeleMed / FullMed) with sum-assured bands
  3. Financial underwriting thresholds (income multiples, documentation)
  4. Smoker / non-smoker rules
  5. Pre-existing disease (PED) disclosure handling

Style:
  - Markdown, level-2 headings per section.
  - 300-400 words total.
  - Plain, agent-facing language; no actuarial jargon beyond "sum assured" and "PED".
  - Where you assume a numeric threshold, mark it as DRAFT in square brackets.

Return only the markdown document.
"""

resp = client.models.generate_content(
    model=MODEL_ID,
    contents=DEMO_2_PROMPT,
    config=types.GenerateContentConfig(temperature=0.0),
)

uw_doc_md = resp.text.strip()

demo_2_path = OUT_DIR / "demo_2_output.md"
header = (
    f"<!-- Generated by {MODEL_ID} at "
    f"{dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds')} UTC -->\n"
    "<!-- Illustrative — ABC Insurer is hypothetical. Numerical thresholds "
    "are DRAFT until matched to ABC Life's live UW matrix. -->\n\n"
)
demo_2_path.write_text(header + uw_doc_md + "\n", encoding="utf-8")

# Preview the first 25 lines.
print("=== Preview (first 25 lines) ===")
for line in uw_doc_md.splitlines()[:25]:
    print(line)
print("..." if len(uw_doc_md.splitlines()) > 25 else "")

# Word-count check.
wc = len(uw_doc_md.split())
print(f"\nWord count: {wc} (brief was 300-400)")

_log_call("demo_2", DEMO_2_PROMPT, resp.text)
print()
print(f"Output saved to {demo_2_path}")


=== Preview (first 25 lines) ===
## ABC Term Life: Underwriting Guidelines

These guidelines outline the criteria for issuing ABC Term Life policies. Please ensure all applications adhere to these rules for a smooth underwriting process.

### 1. Eligibility & Age Bands

*   **Minimum Issue Age:** 25 years (last birthday).
*   **Maximum Issue Age:** 55 years (last birthday).
*   The policy term chosen must ensure that the policyholder's age at maturity does not exceed [DRAFT] 70 years.

### 2. Medical Underwriting Tiers

The level of medical assessment depends on the Sum Assured requested:

*   **No Medical (NoMed):** For Sums Assured up to INR [DRAFT] 25 Lakhs. No medical tests or tele-interview are typically required.
*   **Tele-Medical (TeleMed):** For Sums Assured above INR [DRAFT] 25 Lakhs and up to INR [DRAFT] 50 Lakhs. A tele-interview will be conducted by a medical professional to assess health details.
*   **Full Medical (FullMed):** For Sums Assured above INR [DRAFT] 50 Lakhs 

## 3. Demo 3 — ABC Health board summary (Health)

**Slide:** S2.P2.8
**Goal:** Take a JSON dict of model results from a hospitalisation-frequency
model and produce a 200-word executive summary suitable for the ABC Health
board pack.
**Output format:** Markdown, ~200 words, saved to `/content/demo_3_output.md`.

The pattern: structured numbers in, structured prose out. No statistic in the
output that wasn't in the input.


In [9]:
# --- Demo 3 ---------------------------------------------------------------
# Hypothetical model results dict — this is what the actuary's pipeline would emit.
model_results = {
    "model":           "Hospitalisation frequency, ABC Health 2024",
    "type":            "XGBoost",
    "n_policies":      3200,
    "n_members":       7678,
    "test_AUC":        0.78,
    "lift_top_decile": 3.4,
    "top_features":    ["member_age", "PED_flag", "sum_insured", "region", "product_tier"],
    "fairness": {
        "M_F_diff_pp":  2.1,
        "tier_gap_pp":  0.8,
    },
    "training_window": "2023-01 to 2024-09",
    "test_window":     "2024-10 to 2024-12",
    "calibration_OK":  True,
}

DEMO_3_PROMPT = f"""You are writing the executive summary section of an
internal board pack at ABC Health, a hypothetical Indian health insurer.

Source: the JSON below contains the actual results from a hospitalisation-
frequency model built by the analytics team.

JSON:
{json.dumps(model_results, indent=2)}

Write a 200-word executive summary in markdown. Rules:
  - Lead the very first sentence with the business takeaway (the lift number).
  - Explain AUC in one phrase suitable for non-technical board members. Do not
    define it formally.
  - Name the top three feature drivers; group the rest as "other plausible drivers".
  - Report the fairness gap honestly, do not bury it.
  - Use only numbers that appear in the JSON. Invent nothing.
  - End with a one-line recommendation on whether to deploy.

Return only the markdown summary.
"""

resp = client.models.generate_content(
    model=MODEL_ID,
    contents=DEMO_3_PROMPT,
    config=types.GenerateContentConfig(temperature=0.0),
)

summary_md = resp.text.strip()

demo_3_path = OUT_DIR / "demo_3_output.md"
header = (
    f"<!-- Generated by {MODEL_ID} at "
    f"{dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds')} UTC -->\n"
    "<!-- Illustrative — ABC Insurer is hypothetical. -->\n\n"
    "# ABC Health — Hospitalisation Model · Board Summary\n\n"
)
demo_3_path.write_text(header + summary_md + "\n", encoding="utf-8")

print("=== Generated executive summary ===")
print(summary_md)

wc = len(summary_md.split())
print(f"\nWord count: {wc} (brief was 200)")

_log_call("demo_3", DEMO_3_PROMPT, resp.text)
print()
print(f"Output saved to {demo_3_path}")


=== Generated executive summary ===
The new Hospitalisation frequency model demonstrates a strong business impact, identifying members 3.4 times more likely to be hospitalised in the top decile. Built using XGBoost on 3200 policies covering 7678 members, the model achieved a test AUC of 0.78, indicating good predictive power in distinguishing between members who will and will not be hospitalised.

Key drivers of hospitalisation frequency include member age, pre-existing disease (PED) flag, and sum insured, with region and product tier identified as other plausible drivers. Fairness analysis reveals a 2.1 percentage point difference in predictions between male and female members, and a 0.8 percentage point gap across product tiers. The model's calibration is confirmed as acceptable.

Given its strong predictive performance and acceptable calibration, we recommend deploying this model.

Word count: 124 (brief was 200)

Output saved to /content/demo_3_output.md


## Wrap-up

You should now be able to:

- Call the Gemini API from Colab with a pinned model identifier and an API key
  loaded from Colab Secrets.
- Run a structured-output prompt that returns parseable JSON (Demo 1).
- Run a freeform-markdown prompt with explicit structure, length, and voice
  constraints (Demo 2).
- Feed a JSON of model results into a prompt and produce a board-ready prose
  summary that uses only the numbers you fed in (Demo 3).
- Log every prompt/response pair, with timestamp and model identifier, in a
  one-line CSV record (`/content/prompt_log.csv`).

**Where to next:** open `05_agents_intro.ipynb` for the minimal Agno + Gemini
hello-agent that powers Part 3 of Session 2.

**Companion slides:** S2.P2.4 (Demo 1), S2.P2.6 (Demo 2), S2.P2.8 (Demo 3).

**Validation reminder:** The actuary owns the number, not the model. Before any
output here goes near a regulatory deliverable, verify every figure against a
real source.

**Demonstrated:** three reproducible Gemini prompts (JSON output, markdown UW
draft, board-ready model summary) — all saved to `/content/` with full prompt-log
audit trail.


In [10]:
# One-line summary of what was demonstrated.
print(
    "Demonstrated: three reproducible Gemini prompts "
    "(JSON output, markdown UW draft, board-ready model summary) "
    "saved to /content/ with full prompt-log audit trail."
)


Demonstrated: three reproducible Gemini prompts (JSON output, markdown UW draft, board-ready model summary) saved to /content/ with full prompt-log audit trail.
